In [ ]:
#!pip install pymupdf langchain langchain-community chromadb langchain_huggingface sentence-transformers ollama

In [ ]:
## Cell 1 — Imports
import pdfplumber
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
# from langchain_huggingface import HuggingFaceEmbeddings
import ollama

In [ ]:
# Cell 2 — Load and extract text from PDF
def load_pdf(path: str) -> str:
    with pdfplumber.open(path) as pdf:
        return "\n".join(page.extract_text() for page in pdf.pages)

pdf_text = load_pdf("./data/causal_roadmap.pdf")  # <-- change to your PDF path
print(f"Loaded {len(pdf_text):,} characters")

In [ ]:
# Cell 3 — Split into chunks and build vector store
splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)
chunks = splitter.split_text(pdf_text)
print(f"Created {len(chunks)} chunks")

embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
# embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5") # <- alternmative embeddings model
vectorstore = Chroma.from_texts(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Vector store ready ✓")

In [ ]:
# Cell 4 — Chat function
def chat(question: str, history: list[dict]) -> str:
    # Retrieve relevant chunks from the PDF
    docs = retriever.invoke(question)
    context = "\n\n---\n\n".join(d.page_content for d in docs)

    # Build the prompt
    system_prompt = (
        "You are a helpful assistant. Answer the user's question using ONLY "
        "the context below. If the answer is not in the context, say so.\n\n"
        f"CONTEXT:\n{context}"
    )

    # Append this turn to history
    history.append({"role": "user", "content": question})

    response = ollama.chat(
        model="qwen3.5:9b",   # or "gemma:2b", "gemma:7b", etc.
        messages=[{"role": "system", "content": system_prompt}] + history,
    )

    answer = response["message"]["content"]
    history.append({"role": "assistant", "content": answer})
    return answer

In [ ]:
# Cell 5 — Interactive chat loop (run this cell to start chatting)
history = []
print("PDF Chatbot ready. Type 'quit' to exit.\n")

while True:
    question = input("You: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        break
    if not question:
        continue
    answer = chat(question, history)
    print(f"\nLLM: {answer}\n")

In [7]:
model="qwen3.5:9b"
! ollama stop {model}